"""

Created on Feb 28 2026

by Minde An
mindean@mit.edu


"""

In [ ]:
## In[0]:
# Import necessary packages
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pathlib import Path
from mit_inversions.inversion.inversion import hbmcmc_inversion
from mit_inversions.inversion.mcmc_builder import make_R_prior_sigma, make_x_prior_normal, make_x_prior_mvnormal, make_x_prior_scaling
from mit_inversions.inversion.post_process import (
    build_mcmc_metadata,
    save_mcmc_outputs,
    postprocess_mcmc_outputs,
    postprocess_y_outputs,
    compute_annual_totals,
    plot_state_outputs,
)


In [ ]:
# In[1]:
# Read data at Cape Grim and Trinidad head

# Cape Grim, 41S
df_CGO = pd.read_csv(f'{Path.cwd()}/AGAGE-GCMD_CGO_cfc-11_mon.txt',  sep=r"\s+", comment='#', names=['t', 'year','month', 'cfc11', 'uncertainty','numb'])
df_CGO['date'] = pd.to_datetime(df_CGO[['year', 'month']].assign(day=1))
df_CGO.set_index('date', inplace=True)
df_CGO = df_CGO.drop(columns=['t', 'year', 'month', 'numb'])
df_CGO=df_CGO.resample('YE').mean()


# Trinidad head, 41N
df_THD = pd.read_csv(f'{Path.cwd()}/AGAGE-GCMD_THD_cfc-11_mon.txt',  sep=r"\s+", comment='#', names=['t', 'year','month', 'cfc11', 'uncertainty','numb'])
df_THD['date'] = pd.to_datetime(df_THD[['year', 'month']].assign(day=1))
df_THD.set_index('date', inplace=True)
df_THD = df_THD.drop(columns=['t', 'year', 'month', 'numb'])
df_THD=df_THD.resample('YE').mean()

df_merge = pd.merge(df_CGO, df_THD, on='date', suffixes=('_CGO', '_THD'), how='inner')
df_merge['differences'] = df_merge['cfc11_THD'] - df_merge['cfc11_CGO']
df_merge.index = pd.to_datetime(df_merge.index.year.astype(str) + '-01-01')

del df_CGO, df_THD

In [ ]:
# In[2]:

# Define 3-box model
def boxModel(t,x, E,T_st, T_ts, T_ns, T_L):
    '''
    E: emission rate in each box
    T_st: stratosphere residence time
    T_ts: troposphere residence time
    T_ns: North-South hemisphere exchange time
    T_L: lifetime related to the loss rate in each box
    
    '''
    M = 5.15e21/28.9 #total moles of air
    m = np.array([0.15,0.425,0.425])*M #air moles in each box
    L = np.nan_to_num(1/T_L, nan=0.0) #calculate the loss rate in each box
    
    # mass balance equations
    dxdt=[0,0,0] # covert to ppt
    dxdt[0] = 1/m[0] * ( E[0][int(t)]*1e12  - m[0]*x[0]*L[0] + (m[1]*x[1]+ m[2]*x[2])/T_ts - m[0]*x[0]/T_st )
    dxdt[1] = 1/m[1] * ( E[1][int(t)]*1e12  - m[1]*x[1]*L[1] + m[0]*x[0]*0.5 /T_st - m[1]*x[1]/T_ts + (m[2]*x[2] - m[1]*x[1])/T_ns )
    dxdt[2] = 1/m[2] * ( E[2][int(t)]*1e12  - m[2]*x[2]*L[2] + m[0]*x[0]*0.5 /T_st - m[2]*x[2]/T_ts + (m[1]*x[1] - m[2]*x[2])/T_ns )
    return dxdt

In [ ]:
# In[3]:

## Matrix version 

# estimate emissions for 2008-2022
n_years = 14

# prior emissions [Gg/yr]
N_prior=45 # Gg/yr
S_prior=5
x_error=0.5

xa=np.tile([20,5],n_years).reshape(2*n_years,1) # prior emissions matrix : changes on top of prior
# prior covariance
P = np.zeros((2*n_years,2*n_years))
np.fill_diagonal(P,np.tile([(20*x_error)**2,(5*x_error)**2],n_years))


# Get observation

y = df_merge.loc["2008":"2021",['cfc11_THD','cfc11_CGO']].values.reshape(2*n_years,1)
# obs covariance
R = np.zeros((2*n_years,2*n_years))
np.fill_diagonal(R,(df_merge.loc["2008":"2021",['uncertainty_THD','uncertainty_CGO']].values**2).reshape(2*n_years,1) )

### Calculate Jacobian matrix using 3-box model (NH-SH to NH-emis)

x0=np.array([204, 245, 243])
T_st=1.5
T_ts=8.6
T_ns=1.1
T_L=np.array([7.8, np.nan, np.nan])
tspan = np.array([0,14])
t_eval = np.arange(0,15)
E=np.array([np.zeros_like(t_eval), [N_prior*1e9/137]*(n_years+1), [S_prior*1e9/137]*(n_years+1)])
sol_ref = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E,T_st, T_ts, T_ns, T_L)   )
H0 = np.zeros((2*n_years,2*n_years))

for sens_y in np.arange(0,n_years):
    # for NH
    E_pert_N = E.copy()
    E_pert_N[1][sens_y] *= 1.1
    sol_pert_N = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E_pert_N,T_st, T_ts, T_ns, T_L)   )
    H0[2*sens_y:,2*sens_y] = (np.column_stack((sol_pert_N.y[1,1:]-sol_ref.y[1,1:],sol_pert_N.y[2,1:]-sol_ref.y[2,1:])).flatten()/(N_prior*0.1))[2*sens_y:]

    # for SH
    E_pert_S = E.copy()
    E_pert_S[2][sens_y] *= 1.1
    sol_pert_S = solve_ivp(boxModel, tspan, x0, t_eval=t_eval, method='LSODA', args=(E_pert_S,T_st, T_ts, T_ns, T_L)   )
    H0[2*sens_y:,2*sens_y+1] = (np.column_stack((sol_pert_S.y[1,1:]-sol_ref.y[1,1:],sol_pert_S.y[2,1:]-sol_ref.y[2,1:])).flatten()/(S_prior*0.1))[2*sens_y:]



In [ ]:
y_i = y - sol_ref.y[1:3,1:].flatten('F').reshape(2*n_years,1)

In [ ]:
H0.shape, y_i.shape, R.shape, P.shape, xa.shape

In [ ]:
years = df_merge.loc['2008':'2021'].index.year.astype(str).tolist()
state_index = [f'{year}_{region}' for year in years for region in ['N', 'S']]
obs_index = [f'{year}_{site}' for year in years for site in ['THD', 'CGO']]

H_df = pd.DataFrame(H0, index=obs_index, columns=state_index)
prior_y = sol_ref.y[1:3,1:].flatten('F')
site_labels = ['THD', 'CGO'] * n_years
time_labels = np.repeat(df_merge.loc['2008':'2021'].index.values, 2)
obs_df = pd.DataFrame({
    'obs_y': np.asarray(y, dtype=float).reshape(-1),
    'prior_y': np.asarray(prior_y, dtype=float).reshape(-1),
    'site': site_labels,
    'time': pd.to_datetime(time_labels),
}, index=obs_index)
xa_df = pd.DataFrame(np.asarray(xa, dtype=float).reshape(-1), index=state_index, columns=['xa'])
prior_add_on_df = pd.DataFrame(np.tile([N_prior, S_prior], n_years).astype(float), index=state_index, columns=['prior_add_on'])
output_path = Path.cwd() / 'postprocess_output' / 'test_hbmcmc_inversion'
output_path.mkdir(parents=True, exist_ok=True)


In [ ]:
#R_prior = make_R_prior_sigma(28, 1, name="R_prior")
R_prior = R ##no divergence
#R_prior = np.diag(R) ##no divergence
#

In [ ]:
#x_prior = make_x_prior_normal(28,xa,)
#x_prior = make_x_prior_mvnormal(28,xa, P=P) 
#x_prior = make_x_prior_scaling(28,xa, scaling_prior={"pdf":"truncatednormal","mu":1,"sigma":1,"lower":0})
x_prior = make_x_prior_scaling(28,xa, scaling_prior={"pdf":"normal","mu":1,"sigma":0.5})


In [ ]:
out = hbmcmc_inversion(
    H0,
    y_i,
    R_prior,
    x_prior,
    n_samples=1000,
    n_tune=500,
    n_chains=4,
    target_accept=0.95,
    use_mvnormal_if_matrix=True,
)


In [ ]:
metadata = build_mcmc_metadata(
    x_prior,
    R_prior,
    n_samples=1000,
    n_tune=500,
    n_chains=4,
    target_accept=0.95,
    obs_index=obs_df.index,
    state_index=xa_df.index,
)

save_mcmc_outputs(
    output_path=output_path,
    idata=out,
    H_used=H_df,
    obs_used=obs_df,
    metadata=metadata,
    R_used=R,
    P_used=P,
    xa=xa_df,
    prior_add_on=prior_add_on_df,
)

mcmc_out = postprocess_mcmc_outputs(
    output_path,
    prior_handling_mode='sampled',
    prior_handling_sample_size=2000,
    prior_handling_random_seed=42,
)
results = mcmc_out['results'].copy()
annual = compute_annual_totals(
    results,
    prior_cov_full=mcmc_out['prior_cov_for_sigma'],
    posterior_cov_full=mcmc_out['posterior_cov'],
)
plot_state_outputs(output_path, results, annual)
y_post_out = postprocess_y_outputs(
    output_path,
    state_mode='increment',
    posterior_state_column='x_mean',
)
diag = mcmc_out['diagnostics']
summary = mcmc_out['summary']
diag


In [ ]:
y_plot = y_post_out['y_results'].copy()
y_plot['time'] = pd.to_datetime(y_plot['time'])
gradient_df = y_plot.pivot(index='time', columns='site', values=['obs_y', 'prior_y', 'posterior_y'])
nh_sh_gradient = pd.DataFrame(index=gradient_df.index)
nh_sh_gradient['obs_y'] = gradient_df[('obs_y', 'THD')] - gradient_df[('obs_y', 'CGO')]
nh_sh_gradient['prior_y'] = gradient_df[('prior_y', 'THD')] - gradient_df[('prior_y', 'CGO')]
nh_sh_gradient['posterior_y'] = gradient_df[('posterior_y', 'THD')] - gradient_df[('posterior_y', 'CGO')]

fig = plt.figure()
plt.ylabel('NH-SH gradient (ppt)')
plt.xlabel('Year')
plt.plot(nh_sh_gradient.index, nh_sh_gradient['obs_y'], 'k--', label='Obs NH-SH')
plt.plot(nh_sh_gradient.index, nh_sh_gradient['prior_y'], 'b--', label='Prior Y NH-SH')
plt.plot(nh_sh_gradient.index, nh_sh_gradient['posterior_y'], 'r-', label='Posterior Y NH-SH')
plt.legend()
plt.show()
nh_sh_gradient


In [ ]:
summary


In [ ]:
results


In [ ]:
results_plot = results.copy()
results_plot['year'] = results_plot.index.str.split('_').str[0].astype(int)
results_plot['region'] = results_plot.index.str.split('_').str[1]
plot_df = results_plot.pivot(index='year', columns='region')
year_range = pd.to_datetime(plot_df.index.astype(str) + '-01-01')

fig = plt.figure()
plt.ylabel('CFC-11 Emissions (Gg/yr)')
plt.xlabel('Year')
plt.errorbar(year_range, plot_df['posterior']['N'], yerr=plot_df['state_posterior_sigma']['N'], marker='o', color='b', ecolor='b', label='NH posterior')
plt.errorbar(year_range, plot_df['posterior']['S'], yerr=plot_df['state_posterior_sigma']['S'], marker='o', color='g', ecolor='g', label='SH posterior')
plt.plot(year_range, plot_df['prior']['N'], 'b--', label='NH prior')
plt.plot(year_range, plot_df['prior']['S'], 'g--', label='SH prior')
plt.ylim([0,72])
plt.legend()
plt.show()
